## Rainfall proportion report

In [ ]:
# Global settings
options(scipen=999)

Sys.setenv(PROJ_LIB = "/opt/conda/share/proj")
Sys.setenv(GDAL_DATA = "/opt/conda/share/gdal")

In [ ]:
# Paths
ROOT_PATH <- "~/workspace"
CONFIG_PATH <- file.path(ROOT_PATH, "configuration")
DATA_PATH <- file.path(ROOT_PATH, "data")
OUTPUT_DATA_PATH <- file.path(DATA_PATH, "seasonality_rainfall_proportion")

In [ ]:
# Load utils
source(file.path(ROOT_PATH, "code", "snt_utils.r"))

In [ ]:
# List required packages
required_packages <- c(
  "jsonlite",
  "data.table",
  "arrow",
  "ggplot2",
  "glue",
  "sf",
  "httr",
  "reticulate",
  "gridExtra"
)

install_and_load(required_packages)

# Setup Python/reticulate for openhexa SDK (after installing reticulate)
Sys.setenv(RETICULATE_PYTHON = "/opt/conda/bin/python")
reticulate::py_config()$python
openhexa <- reticulate::import("openhexa.sdk")

In [ ]:
# Load SNT config
CONFIG_FILE_NAME <- "SNT_config.json"
config_json <- tryCatch({ fromJSON(file.path(CONFIG_PATH, CONFIG_FILE_NAME)) },
    error = function(e) {
        msg <- paste0("Error while loading configuration", conditionMessage(e))
        cat(msg)
        stop(msg)
    })

COUNTRY_CODE <- config_json$SNT_CONFIG$COUNTRY_CODE
dhis2_dataset <- config_json$SNT_DATASET_IDENTIFIERS$DHIS2_DATASET_FORMATTED

print(paste("Country code: ", COUNTRY_CODE))

In [ ]:
# Load output data
summary_file <- file.path(OUTPUT_DATA_PATH, glue("{COUNTRY_CODE}_rainfall_proportion.parquet"))

if (!file.exists(summary_file)) {
  stop(glue("File not found: {summary_file}"))
}

summary_dt <- read_parquet(summary_file)
setDT(summary_dt)

In [ ]:
# Quick checks
print(head(summary_dt))
cat("\n=== Summary Statistics ===\n")
cat("MEAN:\n")
print(summary(summary_dt$TOP4_RAINFALL_PROPORTION_MEAN))
cat("\nMEDIAN:\n")
print(summary(summary_dt$TOP4_RAINFALL_PROPORTION_MEDIAN))
cat("\nDifference (MEAN - MEDIAN):\n")
diff_values <- summary_dt$TOP4_RAINFALL_PROPORTION_MEAN - summary_dt$TOP4_RAINFALL_PROPORTION_MEDIAN
print(summary(diff_values))
cat("\nNumber of districts where MEAN > MEDIAN:", sum(diff_values > 0, na.rm = TRUE), "\n")
cat("Number of districts where MEAN < MEDIAN:", sum(diff_values < 0, na.rm = TRUE), "\n")

In [ ]:
# Load spatial data for mapping
spatial_data_filename <- paste(COUNTRY_CODE, "shapes.geojson", sep = "_")
spatial_data <- get_latest_dataset_file_in_memory(dhis2_dataset, spatial_data_filename)
setDT(summary_dt)

# Merge with spatial data
admin_id_col <- "ADM2_ID"
map_data <- merge(spatial_data, summary_dt, by = admin_id_col, all.x = TRUE)

# Function to create proportion categories
create_proportion_cat <- function(values) {
  cut(
    values,
    breaks = c(-Inf, 0, 0.2, 0.4, 0.6, 0.8, 1.0, Inf),
    labels = c("< 0%", "0 - 20%", "20 - 40%", "40 - 60%", "60 - 80%", "80 - 100%", "> 100%"),
    include.lowest = TRUE
  )
}

# Color palette (similar to example)
color_palette <- c(
  "< 0%" = "#E3F2FD",
  "0 - 20%" = "#1976D2",
  "20 - 40%" = "#66BB6A",
  "40 - 60%" = "#FFF9C4",
  "60 - 80%" = "#FFA726",
  "80 - 100%" = "#E65100",
  "> 100%" = "#B71C1C"
)

# Convert to sf if not already
if (!inherits(map_data, "sf")) {
  map_data <- st_as_sf(map_data)
}

# Create categories for both mean and median
map_data$PROPORTION_CAT_MEAN <- create_proportion_cat(map_data$TOP4_RAINFALL_PROPORTION_MEAN)
map_data$PROPORTION_CAT_MEDIAN <- create_proportion_cat(map_data$TOP4_RAINFALL_PROPORTION_MEDIAN)

# Function to create a map
create_map <- function(data, fill_var, title_text) {
  ggplot(data) +
    geom_sf(aes(fill = get(fill_var)), color = "white", size = 0.1) +
    scale_fill_manual(
      values = color_palette,
      name = "Proportion",
      na.value = "grey90",
      drop = FALSE
    ) +
    theme_void() +
    theme(
      legend.position = "bottom",
      plot.title = element_text(hjust = 0.5, size = 12, face = "bold")
    ) +
    labs(
      title = title_text,
      fill = "Proportion"
    )
}

# Create both maps
map_mean <- create_map(map_data, "PROPORTION_CAT_MEAN", "MEAN across years")
map_median <- create_map(map_data, "PROPORTION_CAT_MEDIAN", "MEDIAN across years")

# Display maps side by side
grid.arrange(map_mean, map_median, ncol = 2)

In [ ]:
# Summary of consecutive months
if ("TOP4_RAINFALL_MONTHS_CONSECUTIVE_PROP" %in% names(summary_dt)) {
  cat("\n=== Consecutive Months Analysis ===\n")
  cat("Proportion of years with consecutive top months (by district):\n")
  print(summary(summary_dt$TOP4_RAINFALL_MONTHS_CONSECUTIVE_PROP))
  
  # Districts where months are always consecutive
  always_consecutive <- summary_dt[TOP4_RAINFALL_MONTHS_CONSECUTIVE_PROP == 1, .N]
  cat("\nDistricts where top months are ALWAYS consecutive:", always_consecutive, "\n")
  
  # Districts where months are never consecutive
  never_consecutive <- summary_dt[TOP4_RAINFALL_MONTHS_CONSECUTIVE_PROP == 0, .N]
  cat("Districts where top months are NEVER consecutive:", never_consecutive, "\n")
}

In [ ]:
# Distribution comparison: Mean vs Median
library(ggplot2)

# Prepare data for comparison plot
comparison_dt <- data.table(
  District = rep(1:nrow(summary_dt), 2),
  Value = c(summary_dt$TOP4_RAINFALL_PROPORTION_MEAN, summary_dt$TOP4_RAINFALL_PROPORTION_MEDIAN),
  Type = rep(c("MEAN", "MEDIAN"), each = nrow(summary_dt))
)

# Histogram comparison
p1 <- ggplot(summary_dt, aes(x = TOP4_RAINFALL_PROPORTION_MEAN)) +
  geom_histogram(bins = 20, fill = "steelblue", alpha = 0.7, color = "white") +
  theme_minimal() +
  labs(
    title = "Distribution: MEAN",
    x = "Proportion",
    y = "Number of districts"
  )

p2 <- ggplot(summary_dt, aes(x = TOP4_RAINFALL_PROPORTION_MEDIAN)) +
  geom_histogram(bins = 20, fill = "darkorange", alpha = 0.7, color = "white") +
  theme_minimal() +
  labs(
    title = "Distribution: MEDIAN",
    x = "Proportion",
    y = "Number of districts"
  )

grid.arrange(p1, p2, ncol = 2)

# Scatter plot: Mean vs Median
ggplot(summary_dt, aes(x = TOP4_RAINFALL_PROPORTION_MEDIAN, y = TOP4_RAINFALL_PROPORTION_MEAN)) +
  geom_point(alpha = 0.6, color = "steelblue") +
  geom_abline(intercept = 0, slope = 1, linetype = "dashed", color = "red") +
  theme_minimal() +
  labs(
    title = "MEAN vs MEDIAN comparison",
    subtitle = "Red line: perfect equality (MEAN = MEDIAN)",
    x = "MEDIAN",
    y = "MEAN"
  ) +
  coord_fixed()